# Talk to Your Knowledge Graph

This notebook walks you through building a small biomedical knowledge graph from real public data sources, then querying it - first using SPARQL directly, then using natural language via an LLM.

## What is a knowledge graph?

A knowledge graph is a way of representing facts as a network of connected entities. Each fact is stored as a **triple** — a subject, a predicate, and an object:

```
ERLOTINIB  →  targets  →  EGFR
EGFR       →  associatedWith  →  lung cancer
EGFR       →  participatesIn  →  RAF/MAP kinase cascade
```

This format, called **RDF (Resource Description Framework)**, lets you combine data from multiple sources into a single queryable network, as long as shared identifiers are used to link them.

## What you will build

By the end of this notebook you will have:
- Retrieved real biological data from three public databases (Reactome, ChEMBL, Open Targets)
- Built an RDF knowledge graph connecting drugs, proteins, diseases, and a pathway
- Visualized part of the graph
- Queried it using SPARQL
- Asked it questions in plain English using an LLM

## Data flow

| Stage | Input | Output |
|---|---|---|
| Fetch pathway | Reactome API | `data_raw/reactome_participants.json` |
| Extract proteins | raw participants | `data_processed/proteins.json` |
| Fetch drug targets | ChEMBL API | `data_raw/chembl_mechanisms.json` |
| Fetch disease associations | Open Targets API | `data_raw/opentargets_diseases.json` |
| Build graph | all processed data | `data_processed/graph.ttl` |

## A note on scope

This is a tutorial graph, simplified for clarity and tractability. We focus on a single well-characterised signalling pathway (the RAF/MAP kinase cascade) and subset the data to proteins only, keeping the graph manageable while remaining grounded in real biology. The data sources, identifiers, and graph construction patterns used here are intended to be representative of production biomedical knowledge graph practice.

In [ ]:
import requests
import json
from pathlib import Path
from tqdm.notebook import tqdm

# Reactome pathway (example: RAF/MAP kinase cascade / R-HSA-5673001)
PATHWAY_ID = "R-HSA-5673001"

# Reactome API
url = f"https://reactome.org/ContentService/data/participants/{PATHWAY_ID}"

print("Fetching Reactome participants...")

data = requests.get(url).json()

Path("data_raw").mkdir(exist_ok=True)

with open("data_raw/reactome_participants.json", "w") as f:
    json.dump(data, f, indent=2)

print("Saved to data_raw/reactome_participants.json")
print("Participants:", len(data))

## Stage 1 — Fetch Pathway Participants from Reactome

[Reactome](https://reactome.org) is a curated database of biological pathways: sequences of molecular events involved in processes like signalling, metabolism, and gene regulation.

We will use the **RAF/MAP kinase cascade** (Reactome ID: `R-HSA-5673001`) as our anchor pathway. This is a well-characterised signalling pathway frequently mutated in cancers such as melanoma.

Reactome exposes a public REST API. The endpoint used here returns every **participant** in a given pathway, including proteins, complexes, small molecules, and other biological entities. Each participant includes references to external databases (such as UniProt for proteins), which we will use in the next step.

The raw response is saved to `data_raw/` without modification. Throughout this notebook, `data_raw/` holds unprocessed API responses and `data_processed/` holds cleaned, derived outputs.

## Inspecting the raw data

The response is a list of participant objects in JSON format. Each participant has a `refEntities` field containing references to external databases. You will notice entries of type `CandidateSet` — these represent groups of proteins or other entities that collectively perform a function in the pathway, for example as part of a larger complex. We will filter these out in the next step, keeping only entries with a confirmed UniProt protein identifier.

In [ ]:
with open("data_raw/reactome_participants.json") as f:
    data=json.load(f)

data[:2]

## Stage 2 — Extract Protein Identifiers

From the raw participant data we extract only entries that have a confirmed **UniProt accession** — the standard identifier for proteins in the UniProt database (e.g. `P00533` for EGFR). We identify these by checking that the `stId` field starts with `"uniprot:"`.

Focusing on proteins keeps the graph tractable and makes the downstream queries easier to reason about. Other entity types (small molecules, RNA, complexes) are discarded at this stage.

The cleaned list is saved to `data_processed/proteins.json` and will be used as the input for both the ChEMBL and Open Targets queries in the next two stages.

In [ ]:
import json
from pathlib import Path

with open("data_raw/reactome_participants.json") as f:
    participants = json.load(f)

proteins = {}

for p in participants:
    for ref in p.get("refEntities", []):
        st_id = ref.get("stId", "")
        identifier = ref.get("identifier")
        name = ref.get("displayName")

        if st_id.startswith("uniprot:") and identifier:
            proteins[identifier] = name

protein_list = [
    {
        "uniprot_accession": acc,
        "name": name
    }
    for acc, name in proteins.items()
]

protein_list = sorted(protein_list, key=lambda x: x["uniprot_accession"])

Path("data_processed").mkdir(exist_ok=True)

with open("data_processed/proteins.json", "w") as f:
    json.dump(protein_list, f, indent=2)

print("Proteins extracted:", len(protein_list))
print("Saved to data_processed/proteins.json")

## Inspect the extracted proteins

Each entry contains a UniProt accession and a display name. The display name follows the format `UniProt:ACCESSION GENESYMBOL` — we will use the gene symbol component later when querying Open Targets.

In [ ]:
with open("data_processed/proteins.json") as f:
    data=json.load(f)

data[:20]

## Stage 3 — Fetch Drug-Target Mechanisms from ChEMBL

[ChEMBL](https://www.ebi.ac.uk/chembl/) is a database of bioactive molecules and their known interactions with biological targets. For each protein in our list we query ChEMBL to find drugs with a known **mechanism of action** against that target — for example, ERLOTINIB is an inhibitor of EGFR.

This requires three sequential API calls per protein:

1. Look up the ChEMBL target ID using the UniProt accession
2. Retrieve mechanism records for that target (drug + action type)
3. Fetch the preferred drug name for each molecule

Results are saved to `data_raw/chembl_mechanisms.json`. This step takes approximately 10 minutes due to the number of API calls.

In [ ]:
import json
from pathlib import Path
import requests

# load clean protein list previous step
with open("data_processed/proteins.json") as f:
    proteins = json.load(f)

uniprot_ids = sorted({p["uniprot_accession"] for p in proteins})

print("Unique UniProt accessions:", len(uniprot_ids))
print(uniprot_ids[:10])

results = []
seen = set()

print("Querying ChEMBL using UniProt accessions.  This will take ~10 minutes...")

for accession in tqdm(uniprot_ids):

    target_url = (
        "https://www.ebi.ac.uk/chembl/api/data/target.json"
        f"?target_components__accession={accession}"
        "&target_type=SINGLE%20PROTEIN"
    )

    try:
        target_data = requests.get(target_url).json()
    except Exception as e:
        print(f"  Error fetching target for {accession}: {e}")
        continue

    for target in target_data.get("targets", []):
        target_id = target.get("target_chembl_id")
        target_name = target.get("pref_name")

        mech_url = (
            "https://www.ebi.ac.uk/chembl/api/data/mechanism.json"
            f"?target_chembl_id={target_id}"
        )

        try:
            mech_data = requests.get(mech_url).json()
        except Exception as e:
            print(f"  Error fetching mechanisms for {target_id}: {e}")
            continue

        for mech in mech_data.get("mechanisms", []):
            mol_id = mech.get("molecule_chembl_id")

            if not mol_id:
                continue

            mol_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/{mol_id}.json"

            try:
                mol_data = requests.get(mol_url).json()
            except Exception as e:
                print(f"  Error fetching molecule {mol_id}: {e}")
                continue

            drug_name = mol_data.get("pref_name")

            record_key = (
                accession,
                target_id,
                mol_id,
                mech.get("action_type"),
                mech.get("mechanism_of_action"),
            )

            if record_key in seen:
                continue

            seen.add(record_key)

            results.append({
                "uniprot_accession": accession,
                "target_chembl_id": target_id,
                "target_name": target_name,
                "molecule_chembl_id": mol_id,
                "drug_name": drug_name,
                "action_type": mech.get("action_type"),
                "mechanism_of_action": mech.get("mechanism_of_action"),
            })

Path("data_raw").mkdir(exist_ok=True)

with open("data_raw/chembl_mechanisms.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved to data_raw/chembl_mechanisms.json")
print("Drug-target mechanism records:", len(results))

## Inspect the ChEMBL output

Each record links a UniProt accession to a drug, its action type, and its mechanism of action.

In [ ]:
with open("data_raw/chembl_mechanisms.json") as f:
    data=json.load(f)

data[:5]

## Stage 4 — Fetch Gene-Disease Associations from Open Targets

[Open Targets](https://www.opentargets.org) aggregates evidence from genetics, genomics, literature, and clinical data to score associations between genes and diseases.

Unlike the REST APIs used in the previous stages, Open Targets uses **GraphQL** — rather than hitting separate endpoints, you send a query document specifying exactly the fields you want back. This requires two queries per protein:

1. Search by gene symbol to retrieve the Open Targets target ID (an Ensembl gene ID)
2. Query disease associations for that target, returning the top 10 by association score

The association score is a 0-1 value reflecting the strength and breadth of evidence linking a gene to a disease. Results are saved to `data_raw/opentargets_diseases.json`. This step takes approximately 3 minutes.

In [ ]:
GRAPHQL_URL = "https://api.platform.opentargets.org/api/v4/graphql"

with open("data_processed/proteins.json") as f:
    proteins = json.load(f)

search_query = """
query searchTarget($queryString: String!) {
  search(queryString: $queryString, entityNames: ["target"]) {
    hits {
      id
      entity
      object {
        ... on Target {
          id
          approvedSymbol
          approvedName
        }
      }
    }
  }
}
"""

assoc_query = """
query targetDiseases($ensemblId: String!) {
  target(ensemblId: $ensemblId) {
    id
    approvedSymbol
    associatedDiseases(page: {index: 0, size: 10}) {
      rows {
        disease {
          id
          name
        }
        score
      }
    }
  }
}
"""

results = []

print("Querying Open Targets disease associations using UniProt accessions.  This will take ~3 minutes...")

for protein in tqdm(proteins):
    accession = protein["uniprot_accession"]
    name = protein["name"]

    # name looks like: "UniProt:P00533 EGFR"
    symbol = name.split()[-1]

    try:
        r = requests.post(
            GRAPHQL_URL,
            json={
                "query": search_query,
                "variables": {"queryString": symbol}
            }
        )
        search_data = r.json()
    except Exception as e:
        print(f"  Error searching for {symbol}: {e}")
        continue

    hits = search_data.get("data", {}).get("search", {}).get("hits", [])

    if not hits:
        print("  No target found")
        continue

    hit = hits[0]
    target_obj = hit.get("object", {})
    ensembl_id = target_obj.get("id")
    approved_symbol = target_obj.get("approvedSymbol")

    if not ensembl_id:
        print("  No Ensembl ID found")
        continue

    try:
        r = requests.post(
            GRAPHQL_URL,
            json={
                "query": assoc_query,
                "variables": {"ensemblId": ensembl_id}
            }
        )
        assoc_data = r.json()
    except Exception as e:
        print(f"  Error fetching associations for {ensembl_id}: {e}")
        continue

    rows = (
        assoc_data.get("data", {})
        .get("target", {})
        .get("associatedDiseases", {})
        .get("rows", [])
    )

    for row in rows:
        disease = row.get("disease", {})
        disease_id = disease.get("id")
        disease_name = disease.get("name")
        score = row.get("score")

        results.append({
            "uniprot_accession": accession,
            "gene_symbol": approved_symbol,
            "target_id": ensembl_id,
            "disease_id": disease_id,
            "disease_name": disease_name,
            "association_score": score
        })

Path("data_raw").mkdir(exist_ok=True)

with open("data_raw/opentargets_diseases.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved to data_raw/opentargets_diseases.json")
print("Protein-disease records:", len(results))

## Stage 5 — Build the RDF Knowledge Graph

This is the core of the notebook. We combine all three data sources into a single RDF graph using the `rdflib` library.

### Recap: triples and URIs

As introduced at the top of the notebook, RDF represents knowledge as triples:

```
subject  →  predicate  →  object
```

Every entity in an RDF graph is identified by a **URI** — a web address used as a globally unique identifier. For example, the protein EGFR is identified as:

```
http://purl.uniprot.org/uniprot/P00533
```

URIs don't need to resolve to a real webpage — they just need to be unique and consistent across your graph.

### Namespaces and prefixes

Rather than writing full URIs everywhere, RDF uses **namespaces** — base URI patterns that get a short prefix alias:

```python
UNIPROT = Namespace("http://purl.uniprot.org/uniprot/")
# UNIPROT["P00533"] expands to http://purl.uniprot.org/uniprot/P00533
```

The `g.bind()` calls register these prefixes with the graph so that when the graph is written to file, URIs appear in compact form (e.g. `uniprot:P00533`) rather than as full URLs.

The `ex:` prefix (short for "example") is used for terms we define ourselves — classes like `ex:Drug` and predicates like `ex:targets` — where no existing standard term fits. In a production graph you would reuse established ontology terms where possible.

### Schema hints

Before adding data, we declare the structure of the graph:

- **Classes** (`ex:Drug`, `ex:Protein` etc.) — the types of entity that exist
- **Properties** (`ex:targets`, `ex:associatedWith` etc.) — the relationship types between entities
- **Domain and range** — for each property, what type of entity is the subject (domain) and what type is the object (range)

These declarations are optional — the graph functions without them — but they make the graph self-describing and are useful for any tool that needs to understand its structure.

### How the three data sources connect

The **UniProt accession** is the shared key that stitches all three sources together. The same URI is constructed in each of the three population loops:

```python
protein_uri = UNIPROT[accession]  # identical in all three loops
```

Because the URI is identical, `rdflib` recognises it as the same node across all three loops. By the end you have:

```
drug  →  targets  →  protein  →  associatedWith  →  disease
                     protein  →  participatesIn  →  pathway
```

### Build order

The graph is populated sequentially:

1. **Proteins** — nodes created and connected to the pathway
2. **Drugs** — nodes created and connected to existing protein nodes
3. **Diseases** — nodes created and connected to existing protein nodes

The protein is the hub that everything else connects through.

In [ ]:
import json
from pathlib import Path
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS
from rdflib.namespace import SKOS

# ----------------------------
# load input data
# ----------------------------

with open("data_processed/proteins.json") as f:
    proteins = json.load(f)

with open("data_raw/chembl_mechanisms.json") as f:
    chembl = json.load(f)

with open("data_raw/opentargets_diseases.json") as f:
    diseases = json.load(f)

# ----------------------------
# initialize graph
# ----------------------------

g = Graph()

EX = Namespace("http://example.org/")
UNIPROT = Namespace("http://purl.uniprot.org/uniprot/")
CHEMBL_MOL = Namespace("https://www.ebi.ac.uk/chembl/compound/")
CHEMBL_TARGET = Namespace("https://www.ebi.ac.uk/chembl/target_report_card/")
REACTOME = Namespace("https://reactome.org/content/detail/")
EFO = Namespace("http://www.ebi.ac.uk/efo/")
DOID = Namespace("http://purl.obolibrary.org/obo/DOID_")

g.bind("ex", EX)
g.bind("uniprot", UNIPROT)
g.bind("chemblmol", CHEMBL_MOL)
g.bind("chembltarget", CHEMBL_TARGET)
g.bind("reactome", REACTOME)
g.bind("efo", EFO)
g.bind("doid", DOID)
g.bind("rdfs", RDFS)
g.bind("skos", SKOS)

# ----------------------------
# ontology / schema hints
# ----------------------------

g.add((EX.Drug, RDF.type, RDFS.Class))
g.add((EX.Protein, RDF.type, RDFS.Class))
g.add((EX.Pathway, RDF.type, RDFS.Class))
g.add((EX.Disease, RDF.type, RDFS.Class))
g.add((EX.Target, RDF.type, RDFS.Class))

g.add((EX.Drug, RDFS.label, Literal("Drug")))
g.add((EX.Protein, RDFS.label, Literal("Protein")))
g.add((EX.Pathway, RDFS.label, Literal("Pathway")))
g.add((EX.Disease, RDFS.label, Literal("Disease")))
g.add((EX.Target, RDFS.label, Literal("Target")))

g.add((EX.targets, RDF.type, RDF.Property))
g.add((EX.associatedWith, RDF.type, RDF.Property))
g.add((EX.participatesIn, RDF.type, RDF.Property))
g.add((EX.actsOnTarget, RDF.type, RDF.Property))
g.add((EX.actionType, RDF.type, RDF.Property))
g.add((EX.mechanismOfAction, RDF.type, RDF.Property))
g.add((EX.associationScore, RDF.type, RDF.Property))
g.add((EX.uniprotAccession, RDF.type, RDF.Property))

g.add((EX.targets, RDFS.domain, EX.Drug))
g.add((EX.targets, RDFS.range, EX.Protein))

g.add((EX.associatedWith, RDFS.domain, EX.Protein))
g.add((EX.associatedWith, RDFS.range, EX.Disease))

g.add((EX.participatesIn, RDFS.domain, EX.Protein))
g.add((EX.participatesIn, RDFS.range, EX.Pathway))

g.add((EX.actsOnTarget, RDFS.domain, EX.Drug))
g.add((EX.actsOnTarget, RDFS.range, EX.Target))

# ----------------------------
# pathway
# ----------------------------

pathway_id = "R-HSA-5673001"
pathway_name = "RAF/MAP kinase cascade"
pathway_uri = REACTOME[pathway_id]

g.add((pathway_uri, RDF.type, EX.Pathway))
g.add((pathway_uri, RDFS.label, Literal(pathway_name)))

# ----------------------------
# protein aliases
# ----------------------------

protein_aliases = {
    "EGFR": ["ERBB1", "HER1"],
    "BRAF": ["B-RAF"],
    "MAPK1": ["ERK2"],
    "MAPK3": ["ERK1"]
}

# ----------------------------
# disease aliases
# ----------------------------

disease_aliases = {
    "melanoma": ["skin melanoma", "cutaneous melanoma"]
}

# ----------------------------
# proteins
# ----------------------------

for p in proteins:

    accession = p["uniprot_accession"]
    full_name = p["name"]

    short_name = full_name.split()[-1] if full_name else accession

    protein_uri = UNIPROT[accession]

    g.add((protein_uri, RDF.type, EX.Protein))
    g.add((protein_uri, RDFS.label, Literal(short_name)))
    g.add((protein_uri, EX.uniprotAccession, Literal(accession)))
    g.add((protein_uri, EX.participatesIn, pathway_uri))

    if short_name in protein_aliases:
        for alias in protein_aliases[short_name]:
            g.add((protein_uri, SKOS.altLabel, Literal(alias)))

# ----------------------------
# drugs
# ----------------------------

for row in chembl:

    accession = row["uniprot_accession"]
    mol_id = row["molecule_chembl_id"]
    drug_name = row.get("drug_name")
    action_type = row.get("action_type")
    mechanism = row.get("mechanism_of_action")
    target_id = row.get("target_chembl_id")
    target_name = row.get("target_name")

    if not mol_id:
        continue

    protein_uri = UNIPROT[accession]
    drug_uri = CHEMBL_MOL[mol_id]

    g.add((drug_uri, RDF.type, EX.Drug))
    g.add((drug_uri, EX.targets, protein_uri))

    if drug_name:
        g.add((drug_uri, RDFS.label, Literal(drug_name)))

    if action_type:
        g.add((drug_uri, EX.actionType, Literal(action_type)))

    if mechanism:
        g.add((drug_uri, EX.mechanismOfAction, Literal(mechanism)))

    if target_id:
        target_uri = CHEMBL_TARGET[target_id]
        g.add((target_uri, RDF.type, EX.Target))
        g.add((drug_uri, EX.actsOnTarget, target_uri))

        if target_name:
            g.add((target_uri, RDFS.label, Literal(target_name)))

# ----------------------------
# diseases
# ----------------------------

for row in diseases:

    accession = row["uniprot_accession"]
    disease_id = row.get("disease_id")
    disease_name = row.get("disease_name")
    score = row.get("association_score")

    if not disease_id:
        continue

    protein_uri = UNIPROT[accession]

    if disease_id.startswith("EFO_"):
        disease_uri = EFO[disease_id]
    elif disease_id.startswith("DOID_"):
        disease_uri = DOID[disease_id.replace("DOID_", "")]
    else:
        disease_uri = EX[disease_id]

    g.add((disease_uri, RDF.type, EX.Disease))
    g.add((protein_uri, EX.associatedWith, disease_uri))

    if disease_name:
        g.add((disease_uri, RDFS.label, Literal(disease_name)))

        key = disease_name.lower()
        if key in disease_aliases:
            for alias in disease_aliases[key]:
                g.add((disease_uri, SKOS.altLabel, Literal(alias)))

    if score is not None:
        g.add((protein_uri, EX.associationScore, Literal(score)))

# ----------------------------
# output
# ----------------------------

Path("data_processed").mkdir(exist_ok=True)

triples_path = "data_processed/graph_triples.tsv"

with open(triples_path, "w") as f:
    for s, p, o in g:
        f.write(f"{s}\t{p}\t{o}\n")

print(f"Saved raw triples to {triples_path}")

output_path = "data_processed/graph.ttl"
g.serialize(destination=output_path, format="turtle")

print(f"Saved RDF graph to {output_path}")
print(f"Total triples: {len(g)}")

## Inspect the raw triples

The graph is serialized as a tab-separated file for inspection. Each row is one triple — subject, predicate, object — with full URIs.

In [ ]:
import pandas as pd

df = pd.read_csv("data_processed/graph_triples.tsv", sep="\t", names=["subject","predicate","object"])
df.head(10)

## Inspect the Turtle output

The same graph serialized in **Turtle format** — a compact, human-readable RDF notation that applies the prefix bindings to shorten URIs and groups triples by subject. This is the file loaded in the querying section below.

In [ ]:
with open("data_processed/graph.ttl") as f:
    lines = f.readlines()
print("".join(lines[:20]))

# Querying the Knowledge Graph

With the graph built and saved, we now load it back into memory and query it. We will cover two approaches:

1. **Direct SPARQL** — hand-written queries against the graph
2. **Natural language** — questions converted to SPARQL by an LLM, with results summarised in plain English

## What is SPARQL?

**SPARQL** is the standard query language for RDF graphs — roughly analogous to SQL for relational databases. A SPARQL query describes a pattern of triples to match against the graph and returns the variables that satisfy that pattern.

## Prerequisites

- `data_processed/graph.ttl` must exist (run all cells above first)
- An OpenAI API key set as the environment variable `OPENAI_API_KEY`

In [ ]:
from pathlib import Path
from rdflib import Graph

GRAPH_PATH = Path("data_processed/graph.ttl")
assert GRAPH_PATH.exists(), f"Could not find {GRAPH_PATH}"

g = Graph()
g.parse(GRAPH_PATH, format="turtle")
print(f"Loaded triples: {len(g):,}")

In [ ]:
PREFIXES = """
PREFIX ex: <http://example.org/>
PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
PREFIX chemblmol: <https://www.ebi.ac.uk/chembl/compound/>
PREFIX chembltarget: <https://www.ebi.ac.uk/chembl/target_report_card/>
PREFIX reactome: <https://reactome.org/content/detail/>
PREFIX efo: <http://www.ebi.ac.uk/efo/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
"""

print(PREFIXES)

SPARQL queries are plain strings — they don't have access to the prefix bindings registered with the graph object. We define the prefixes here as a string constant to prepend to every query.

## Visualizing the Graph

Before querying the full graph, we build a small subgraph to visualize. We select 25 drug-protein pairs using a SPARQL query and render them as an interactive network using `pyvis`. This gives an intuitive feel for the graph structure before we move to text-based querying.

In [ ]:
import networkx as nx

G = nx.DiGraph()

query = PREFIXES + """
SELECT ?drugLabel ?proteinLabel
WHERE {
  ?drug ex:targets ?protein ;
        rdfs:label ?drugLabel .
  ?protein rdfs:label ?proteinLabel .
}
LIMIT 25
"""

for drug, protein in g.query(query):
    G.add_node(drug, type="drug")
    G.add_node(protein, type="protein")
    G.add_edge(drug, protein)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

The cell below renders an interactive HTML visualization. Proteins are shown in blue, drugs in red. You can drag nodes, zoom, and pan to explore the network.

> **Note:** This visualization requires JupyterLab. It will not render in VSCode or other notebook editors.

In [ ]:
from pyvis.network import Network
from IPython.display import display, HTML

net = Network(notebook=True, directed=True, cdn_resources='in_line')

for node in G.nodes:
    node_type = G.nodes[node]["type"]
    color = "red" if node_type == "drug" else "blue"
    net.add_node(node, label=node, color=color)

for source, target in G.edges:
    net.add_edge(source, target)

display(HTML(net.generate_html()))

## Direct SPARQL Queries

Now we query the full graph using hand-written SPARQL. Each query follows the same basic structure:

```sparql
SELECT ?variable
WHERE {
  ?subject  predicate  ?object .
}
```

- Variables are prefixed with `?`
- `;` is shorthand for "same subject, new predicate"
- `|` matches either of two predicates — used here to match both `rdfs:label` and `skos:altLabel` so queries work across preferred names and synonyms
- Patterns chain together by sharing variables — this is how multi-hop traversal works

The three queries below demonstrate progressively more complex traversals of the graph.

List all proteins that participate in the RAF/MAP kinase cascade pathway.

In [ ]:
q1 = PREFIXES + """
SELECT ?proteinLabel
WHERE {
  ?protein ex:participatesIn reactome:R-HSA-5673001 ;
           rdfs:label ?proteinLabel .
}
ORDER BY ?proteinLabel
LIMIT 20
"""

for row in g.query(q1):
    print(row[0])

Find all drugs that target EGFR. Note the use of `rdfs:label|skos:altLabel` to match both the preferred label and any registered synonyms such as ERBB1 and HER1.

In [ ]:
q2 = PREFIXES + """
SELECT ?drugLabel
WHERE {
  ?drug ex:targets ?protein ;
        rdfs:label ?drugLabel .
  ?protein rdfs:label|skos:altLabel "EGFR" .
}
ORDER BY ?drugLabel
"""

for row in g.query(q2):
    print(row[0])

A two-hop query: find drugs that target any protein associated with melanoma. This traverses drug → protein → disease in a single query.

In [ ]:
q3 = PREFIXES + """
SELECT ?drugLabel ?diseaseLabel
WHERE {
  ?drug ex:targets ?protein ;
        rdfs:label ?drugLabel .
  ?protein ex:associatedWith ?disease .
  ?disease rdfs:label|skos:altLabel "melanoma" ;
           rdfs:label ?diseaseLabel .
}
ORDER BY ?drugLabel
LIMIT 30
"""

for row in g.query(q3):
    print(row)

## LLM-Assisted Querying

The functions below implement a **natural language to SPARQL** pipeline:

1. `generate_sparql()` sends the question to an LLM along with a detailed schema prompt describing the graph's classes, predicates, and prefix conventions. The LLM returns a SPARQL query as structured JSON.
2. `ask_graph()` converts the question to SPARQL, executes it against the graph, then makes a second LLM call to summarise the results in plain English.

The schema prompt is critical. Without it the LLM would have to guess predicate names and prefix conventions, producing queries that fail or return nothing. The explicit instruction to use `rdfs:label|skos:altLabel` for entity lookup is particularly important, as an LLM defaulting to `rdfs:label` only would miss synonym matches entirely.

In [ ]:
import os, json, re
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

SCHEMA = """
You are writing SPARQL for a small biomedical RDF graph.

Use these prefixes exactly:

PREFIX ex: <http://example.org/>
PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
PREFIX chemblmol: <https://www.ebi.ac.uk/chembl/compound/>
PREFIX chembltarget: <https://www.ebi.ac.uk/chembl/target_report_card/>
PREFIX reactome: <https://reactome.org/content/detail/>
PREFIX efo: <http://www.ebi.ac.uk/efo/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

Classes:
- ex:Drug
- ex:Protein
- ex:Pathway
- ex:Disease
- ex:Target

Main predicates:
- ex:targets (Drug -> Protein)
- ex:associatedWith (Protein -> Disease)
- ex:participatesIn (Protein -> Pathway)
- ex:actsOnTarget (Drug -> Target)

Useful labels:
- rdfs:label = preferred label
- skos:altLabel = alias/synonym

Important rule:
When looking up a named entity like EGFR, ERBB1, HER1, melanoma, skin melanoma, or cutaneous melanoma,
use this pattern:
    ?entity rdfs:label|skos:altLabel ?name .
Do not rely only on rdfs:label.

Main pathway:
- reactome:R-HSA-5673001 = RAF/MAP kinase cascade

Return only a SELECT query. Do not wrap the query in markdown code fences.
"""

def generate_sparql(question: str) -> str:
    prompt = f"""{SCHEMA}

Question: {question}
"""

    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "sparql_response",
                "schema": {
                    "type": "object",
                    "properties": {
                        "sparql": {"type": "string"}
                    },
                    "required": ["sparql"],
                    "additionalProperties": False
                }
            }
        }
    )
    sparql = json.loads(resp.choices[0].message.content)["sparql"]
    # strip markdown code fences if the LLM included them
    sparql = re.sub(r'^```[a-z]*\n?', '', sparql.strip())
    sparql = re.sub(r'\n?```$', '', sparql).strip()
    return sparql

In [ ]:
def ask_graph(question: str, summarize: bool = True):
    sparql = generate_sparql(question)
    print("SPARQL:\n")
    print(sparql)
    print("\n---\n")
    rows = list(g.query(sparql))
    print(f"Rows returned: {len(rows)}")
    for row in rows[:10]:
        print(row)

    if not summarize:
        return {"question": question, "sparql": sparql, "rows": rows}

    summary_prompt = f"""Answer the user's question using only these query results.

Question:
{question}

SPARQL:
{sparql}

Rows:
{rows[:25]}

Give a brief answer. If there are no rows, say that no matching results were found in this graph.
"""

    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": summary_prompt}],
        temperature=0.2
    )
    answer = resp.choices[0].message.content
    print("\nAnswer:\n")
    print(answer)
    return {"question": question, "sparql": sparql, "rows": rows, "answer": answer}

## Example Questions

The four examples below demonstrate progressively more interesting queries. Note how the synonym examples (ERBB1, cutaneous melanoma) test whether the alias handling in the graph and schema prompt are working correctly.

In [ ]:
ask_graph("Which drugs target EGFR?")

This query uses a synonym for EGFR. It should return the same results as the previous query if the `skos:altLabel` aliases are working correctly.

In [ ]:
ask_graph("Which drugs target ERBB1?")

"Cutaneous melanoma" is a synonym for melanoma registered via `skos:altLabel`. This tests whether the LLM generates a query that correctly uses the alias pattern.

In [ ]:
ask_graph("Which diseases are associated with cutaneous melanoma?")

In [ ]:
ask_graph("Which drugs target proteins associated with melanoma?")

### Congratulations!

You have now:
- Encountered real biomedical data sources and file formats
- Built an RDF knowledge graph from real biological data
- Visualized graph structure
- Queried your graph using SPARQL
- Used an LLM to query your graph in natural language

## What next?

Try extending the graph with a new annotation type. For example, **gene → phenotype** associations are available from resources such as the [Human Phenotype Ontology (HPO)](https://hpo.jax.org) or [Open Targets](https://www.opentargets.org). To add this you would:

1. Query the API for phenotype associations using the same UniProt accessions
2. Add new triples to the graph using a new predicate (e.g. `ex:associatedWithPhenotype`)
3. Update the schema hints and LLM schema prompt to include the new predicate
4. Write a SPARQL query to test the new relationships